# Predicting California Housing Prices

### A comparison of Gradient Boosting and a Neural Network for tabular regression

![Python](https://img.shields.io/badge/Python-3.10-3776AB?logo=python&logoColor=white)
![scikit-learn](https://img.shields.io/badge/scikit--learn-Regression-F7931E?logo=scikit-learn&logoColor=white)
![pandas](https://img.shields.io/badge/pandas-Data%20Analysis-150458?logo=pandas&logoColor=white)
![Jupyter](https://img.shields.io/badge/Jupyter-Notebook-F37626?logo=jupyter&logoColor=white)
![License](https://img.shields.io/badge/License-MIT-green)

---

**Goal.** Given eight numeric attributes of a California census block group, predict its **median house value**. This is a classic supervised **regression** problem, and it is a good testbed for comparing a tree-based ensemble against a neural network on structured, tabular data.

**Approach.** Load the data, explore it, split it into training and test sets, then train and evaluate two very different model families side by side under identical conditions.

| | |
|---|---|
| **Task** | Supervised regression |
| **Dataset** | California Housing (1990 U.S. Census) — 20,640 samples, 8 features |
| **Target** | Median house value, in units of \$100,000 |
| **Models** | `GradientBoostingRegressor` vs. `MLPRegressor` (neural network) |
| **Best result** | Gradient Boosting — **R² = 0.81**, mean error ≈ \$33.6k |


## Contents

1. [Regression in a Nutshell](#1-regression-in-a-nutshell)
2. [The California Housing Dataset](#2-the-california-housing-dataset)
3. [Exploratory Data Analysis](#3-exploratory-data-analysis)
4. [Feature Selection and Splitting the Data](#4-feature-selection-and-splitting-the-data)
5. [Building the Models](#5-building-the-models)
6. [Evaluating Performance](#6-evaluating-performance)
7. [Results and Conclusion](#7-results-and-conclusion)

---


## 1. Regression in a Nutshell

**Regression** predicts a continuous numeric value by learning the relationship between a set of *input* variables and an *output* variable.

- **Independent variables (features):** the inputs the model learns from — here, things like median income, house age, and average number of rooms in a neighborhood.
- **Dependent variable (target):** the value we want to predict — here, the median house value of that neighborhood.

The model's job is to find a function that maps the features to the target as accurately as possible, so that when it sees a *new* block group it has never encountered, it can estimate a sensible price.

> New to regression? See this overview from [GeeksforGeeks](https://www.geeksforgeeks.org/machine-learning/regression-in-machine-learning/).


## 2. The California Housing Dataset

The dataset comes bundled with scikit-learn via `sklearn.datasets.fetch_california_housing`. It is derived from the **1990 U.S. Census**, where each row is a *block group* (the smallest geographic unit the Census publishes, typically 600–3,000 people).

**Data dictionary**

| Feature | Description |
|---|---|
| `MedInc` | Median income in the block group (tens of thousands of USD) |
| `HouseAge` | Median house age in the block group |
| `AveRooms` | Average number of rooms per household |
| `AveBedrms` | Average number of bedrooms per household |
| `Population` | Block group population |
| `AveOccup` | Average number of household members |
| `Latitude` | Block group latitude |
| `Longitude` | Block group longitude |
| **`MedHouseVal`** | **Target** — median house value (units of \$100,000) |

> Reference: scikit-learn [User Guide](https://scikit-learn.org/stable/user_guide.html).

We start by loading it as a pandas `DataFrame` and confirming its shape — 8 feature columns and one target, across 20,640 samples.


In [2]:
from sklearn.datasets import fetch_california_housing

housing_dataset = fetch_california_housing(as_frame=True)
print(housing_dataset.data.shape, housing_dataset.target.shape)

(20640, 8) (20640,)


In [3]:
housing = housing_dataset.frame # panadas frame

## 3. Exploratory Data Analysis

Before modeling, it pays to understand the data: its scale, its distribution, whether anything is missing, and how the features relate to the target. The next few cells inspect the raw values (`head`), summary statistics (`describe`), column types (`info`), and null values (`isnull`).


In [4]:
housing.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [5]:
housing.describe()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704,2.068558
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532,1.153956
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000,0.149990
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000,1.196000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000,1.797000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000,2.647250
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000,5.000010


In [6]:
housing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 non-null  float64
 5   AveOccup     20640 non-null  float64
 6   Latitude     20640 non-null  float64
 7   Longitude    20640 non-null  float64
 8   MedHouseVal  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB


In [7]:
housing.isnull()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...
20635,False,False,False,False,False,False,False,False,False
20636,False,False,False,False,False,False,False,False,False
20637,False,False,False,False,False,False,False,False,False
20638,False,False,False,False,False,False,False,False,False


**Correlation with the target.** The Pearson correlation matrix shows how strongly each feature moves with `MedHouseVal`. `MedInc` (median income) stands out as the strongest linear signal (≈ 0.69) — unsurprisingly, wealthier neighborhoods tend to have more expensive homes. Most other features carry weaker linear signal on their own, which is one reason non-linear models (a boosted ensemble and a neural network) are worth trying.


In [8]:
housing.corr("pearson", numeric_only=True) # check for correlation with "MedHouseVal" 


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
MedInc,1.000000,-0.119034,0.326895,-0.062040,0.004834,0.018766,-0.079809,-0.015176,0.688075
HouseAge,-0.119034,1.000000,-0.153277,-0.077747,-0.296244,0.013191,0.011173,-0.108197,0.105623
AveRooms,0.326895,-0.153277,1.000000,0.847621,-0.072213,-0.004852,0.106389,-0.027540,0.151948
AveBedrms,-0.062040,-0.077747,0.847621,1.000000,-0.066197,-0.006181,0.069721,0.013344,-0.046701
Population,0.004834,-0.296244,-0.072213,-0.066197,1.000000,0.069863,-0.108785,0.099773,-0.024650
AveOccup,0.018766,0.013191,-0.004852,-0.006181,0.069863,1.000000,0.002366,0.002476,-0.023737
Latitude,-0.079809,0.011173,0.106389,0.069721,-0.108785,0.002366,1.000000,-0.924664,-0.144160
Longitude,-0.015176,-0.108197,-0.027540,0.013344,0.099773,0.002476,-0.924664,1.000000,-0.045967
MedHouseVal,0.688075,0.105623,0.151948,-0.046701,-0.024650,-0.023737,-0.144160,-0.045967,1.000000


## 4. Feature Selection and Splitting the Data

We separate the DataFrame into:

- **`X`** — the eight feature columns (everything except `MedHouseVal`).
- **`y`** — the target column, `MedHouseVal`.

We then hold out **20%** of the data as an untouched **test set** and train on the remaining **80%**. Fixing `random_state=42` makes the split — and therefore every result in this notebook — fully reproducible. The test set is never seen during training, so it gives an honest estimate of how each model generalizes to new neighborhoods.


In [9]:
X = housing.drop(columns="MedHouseVal") # keeping all the features expect for "MedHouseVal" columns
y = housing.MedHouseVal# select our target

In [10]:
X.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [11]:
y

0        4.526
1        3.585
2        3.521
3        3.413
4        3.422
         ...  
20635    0.781
20636    0.771
20637    0.923
20638    0.847
20639    0.894
Name: MedHouseVal, Length: 20640, dtype: float64

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # 80% data for train set and 20% data for test set

## 5. Building the Models

We train two contrasting model families on the same split.

### 5a. Gradient Boosting

`GradientBoostingRegressor` builds an ensemble of shallow decision trees **sequentially**, where each new tree corrects the errors of the ensemble so far. Tree-based models are naturally scale-invariant, so they need no feature scaling and tend to be a very strong baseline on tabular data.


In [13]:
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler

GBR_model = GradientBoostingRegressor(random_state=42)

GBR_model.fit(X_train, y_train)

,loss,'squared_error'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [14]:
GBR_pred = GBR_model.predict(X_test)
print(f"Gradient Boosting predict: {GBR_pred}")

Gradient Boosting predict: [0.50518761 1.09334601 4.24570956 ... 4.68181295 0.85329537 1.96275219]


### 5b. Neural Network (Multi-Layer Perceptron)

`MLPRegressor` is a fully connected feed-forward neural network. Unlike trees, neural networks are **sensitive to feature scale**, so we wrap the model in a `Pipeline` with `StandardScaler`. The scaler standardizes each feature to zero mean and unit variance, and the pipeline guarantees that this transformation is fit on the training data only — preventing information from the test set from leaking into training.


In [15]:
from sklearn.neural_network import MLPRegressor

MLP_pipline = make_pipeline(StandardScaler(), MLPRegressor(random_state=42))

MLP_pipline.fit(X_train, y_train)

,steps,"[('standardscaler', ...), ('mlpregressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,loss,'squared_error'
,hidden_layer_sizes,"(100,)"
,activation,'relu'
,solver,'adam'


In [35]:
MLP_pred = MLP_pipline.predict(X_test)
print(f"Multi-layer Perceptron (MLP) predict: {MLP_pred}")

Multi-layer Perceptron (MLP) predict: [0.44550995 1.10798349 4.7554052  ... 4.86008377 0.73192556 1.7901959 ]


## 6. Evaluating Performance

To compare the models fairly we report five standard regression metrics. Because the target is measured in units of \$100,000, an error of `0.34` corresponds to roughly **\$34,000**.

| Metric | What it measures | Better |
|---|---|---|
| **R²** | Fraction of variance in price explained by the model | Higher (max 1.0) |
| **MAE** | Mean absolute error — average size of the miss | Lower |
| **MSE** | Mean squared error — penalizes large misses heavily | Lower |
| **RMSE** | Root mean squared error — MSE back in price units | Lower |
| **MAPE** | Mean absolute *percentage* error | Lower |

The helper function below prints all five metrics for a given model so the two can be compared at a glance.


In [17]:
from sklearn.metrics import (
    r2_score, 
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    root_mean_squared_error
    )

In [ ]:

    
def Metrics(model_name, y_true, y_pred):
    print(f"{model_name} R-squared (R²) score: {r2_score(y_true, y_pred)}")
    print(f"{model_name} mean_absolute error: {mean_absolute_error(y_true, y_pred)}")
    print(f"{model_name} mean squared error: {mean_squared_error(y_true, y_pred)}")
    print(f"{model_name} mean absolute percentage error: {mean_absolute_percentage_error(y_true, y_pred)}")
    print(f"{model_name} root mean squared error: {root_mean_squared_error(y_true, y_pred)}")

In [50]:
GBR_default_params = Metrics("GradientBoostingRegressor", y_test, GBR_pred)
GBR_default_params

GradientBoostingRegressor, R-squared (R²) score: 0.8102267390771959
GradientBoostingRegressor, mean_absolute error: 0.336365839889655
GradientBoostingRegressor, mean squared error: 0.24868058494329542
GradientBoostingRegressor, mean absolute percentage error: 0.1952542305234306
GradientBoostingRegressor, root mean squared error: 0.498678839478171


In [51]:
MLP_default_params = Metrics("Multi-layer Perceptron", y_test, MLP_pred)
MLP_default_params

Multi-layer Perceptron, R-squared (R²) score: 0.77685679715835
Multi-layer Perceptron, mean_absolute error: 0.3687731647391245
Multi-layer Perceptron, mean squared error: 0.2924088564371284
Multi-layer Perceptron, mean absolute percentage error: 0.21206591987830745
Multi-layer Perceptron, root mean squared error: 0.5407484225008228


### Results

Both models are evaluated on the same untouched test set:

| Metric | Gradient Boosting | Neural Network (MLP) | Winner |
|---|---|---|---|
| **R²** (↑) | **0.810** | 0.777 | Gradient Boosting |
| **MAE** (↓) | **0.336** | 0.369 | Gradient Boosting |
| **RMSE** (↓) | **0.499** | 0.541 | Gradient Boosting |
| **MAPE** (↓) | **19.5%** | 21.2% | Gradient Boosting |
| **MSE** (↓) | **0.249** | 0.292 | Gradient Boosting |

**Reading the numbers.** Gradient Boosting wins on every metric. Its **R² of 0.81** means it explains about 81% of the variation in neighborhood house prices, versus 78% for the neural network. Its **MAE of 0.336** translates to an average absolute error of roughly **\$33,600** — about \$3,000 tighter than the MLP.

**Why?** This is a common outcome on small, purely tabular datasets: gradient-boosted trees capture non-linear feature interactions out of the box and need no tuning to be competitive, whereas neural networks usually need more data and careful tuning (architecture, learning rate, regularization) to pull ahead. The two cells below print each model's full default hyperparameter set — the exact configuration behind these results — which is also the natural starting point for tuning.


In [52]:
MLP_pipline.get_params()

{'memory': None,
 'steps': [('standardscaler', StandardScaler()),
  ('mlpregressor', MLPRegressor(random_state=42))],
 'transform_input': None,
 'verbose': False,
 'standardscaler': StandardScaler(),
 'mlpregressor': MLPRegressor(random_state=42),
 'standardscaler__copy': True,
 'standardscaler__with_mean': True,
 'standardscaler__with_std': True,
 'mlpregressor__activation': 'relu',
 'mlpregressor__alpha': 0.0001,
 'mlpregressor__batch_size': 'auto',
 'mlpregressor__beta_1': 0.9,
 'mlpregressor__beta_2': 0.999,
 'mlpregressor__early_stopping': False,
 'mlpregressor__epsilon': 1e-08,
 'mlpregressor__hidden_layer_sizes': (100,),
 'mlpregressor__learning_rate': 'constant',
 'mlpregressor__learning_rate_init': 0.001,
 'mlpregressor__loss': 'squared_error',
 'mlpregressor__max_fun': 15000,
 'mlpregressor__max_iter': 200,
 'mlpregressor__momentum': 0.9,
 'mlpregressor__n_iter_no_change': 10,
 'mlpregressor__nesterovs_momentum': True,
 'mlpregressor__power_t': 0.5,
 'mlpregressor__random_sta

In [37]:
GBR_model.get_params()

{'alpha': 0.9,
 'ccp_alpha': 0.0,
 'criterion': 'friedman_mse',
 'init': None,
 'learning_rate': 0.1,
 'loss': 'squared_error',
 'max_depth': 3,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'n_estimators': 100,
 'n_iter_no_change': None,
 'random_state': 42,
 'subsample': 1.0,
 'tol': 0.0001,
 'validation_fraction': 0.1,
 'verbose': 0,
 'warm_start': False}

## 7. Results and Conclusion

**Summary.** On the California Housing dataset, a `GradientBoostingRegressor` with default settings clearly outperformed a default `MLPRegressor`, reaching **R² = 0.81** and an average error of about **\$33.6k** per neighborhood — all with a fully reproducible pipeline and no data leakage.

**Key takeaways**

- On small, tabular datasets, gradient-boosted trees are a very hard baseline to beat.
- Neural networks require feature scaling; a `Pipeline` with `StandardScaler` keeps that transformation leak-free.
- A fixed `random_state` and an untouched test set make the comparison honest and reproducible.

**Next steps**

- **Hyperparameter tuning** with cross-validation (`GridSearchCV` / `RandomizedSearchCV`) for both models.
- **Feature engineering**, e.g. `rooms_per_household` or capping the `AveOccup` / `AveRooms` outliers visible in `describe()`.
- **Stronger baselines** such as `HistGradientBoostingRegressor`, `RandomForestRegressor`, or `XGBoost`.
- **Visual diagnostics**: predicted-vs-actual plots, residual analysis, and a geographic price map using `Latitude` / `Longitude`.

---

*Built with Python, pandas, and scikit-learn.*

**Author:** Juan Marquez &nbsp;•&nbsp; [GitHub](https://github.com/Rokku-Okajima) &nbsp;•&nbsp; [LinkedIn](https://www.linkedin.com/in/juan-m-5bb41629a/)
